In [2]:
# Install required packages
!pip install pycryptodome ecdsa bitcoin base58 tqdm torch scikit-learn

import hashlib
import binascii
from Crypto.Hash import RIPEMD160
import ecdsa
import base58
import random
import time
import numpy as np
import pandas as pd
from tqdm import tqdm
import concurrent.futures
import threading
import os
from google.colab import files
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from collections import deque

# Check if GPU is available and set it up
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Set up display and threading for tracking global bests
print_lock = threading.Lock()
global_bests = {}

# Increase sample size dramatically - keep 100,000 examples for better learning
training_data = deque(maxlen=100000)

# Track best examples separately - these are rare and valuable for learning
best_examples = []  # No size limit for the best examples

# Advanced Neural Network Model with improved architecture
class EnhancedHammingPredictor(nn.Module):
    def __init__(self, input_size=256, hidden_sizes=[512, 256, 128, 64]):
        super(EnhancedHammingPredictor, self).__init__()

        # Feature extraction layers
        layers = []
        prev_size = input_size

        for h_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, h_size))
            layers.append(nn.LeakyReLU(0.2))
            layers.append(nn.BatchNorm1d(h_size))
            layers.append(nn.Dropout(0.2))
            prev_size = h_size

        self.feature_extractor = nn.Sequential(*layers)

        # Regression head
        self.regressor = nn.Sequential(
            nn.Linear(hidden_sizes[-1], 32),
            nn.LeakyReLU(0.2),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        features = self.feature_extractor(x)
        return self.regressor(features)

# Initialize enhanced predictor model
predictor = EnhancedHammingPredictor().to(device)
optimizer = optim.Adam(predictor.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=3, factor=0.5)
criterion = nn.MSELoss()

# Flag for tracking if model has been trained
model_trained = False

# Function to extract comprehensive features from a key
def extract_advanced_features(key_int):
    """Extract advanced bit-level features from a key integer"""
    # Basic bit representation (64 bits)
    basic_bits = [(key_int >> i) & 1 for i in range(64)]

    # Extract byte-level features (8 bytes)
    bytes_values = [(key_int >> (i*8)) & 0xFF for i in range(8)]

    # Calculate runs of 0s and 1s
    runs = []
    current_run = 1
    for i in range(1, 64):
        bit = (key_int >> i) & 1
        prev_bit = (key_int >> (i-1)) & 1
        if bit == prev_bit:
            current_run += 1
        else:
            runs.append(current_run)
            current_run = 1
    runs.append(current_run)

    # Fill to fixed size (padding with zeros if needed)
    runs = runs[:32] + [0] * max(0, 32 - len(runs))

    # Count of 1s in different regions
    region_counts = []
    for i in range(8):  # 8 regions of 8 bits each
        region_count = sum([(key_int >> (i*8 + j)) & 1 for j in range(8)])
        region_counts.append(region_count)

    # Parity bits
    parity_bits = [sum([(key_int >> (i*8 + j)) & 1 for j in range(8)]) % 2 for i in range(8)]

    # Byte transitions (XOR between adjacent bytes)
    transitions = []
    for i in range(7):
        byte1 = (key_int >> (i*8)) & 0xFF
        byte2 = (key_int >> ((i+1)*8)) & 0xFF
        transitions.append(bin(byte1 ^ byte2).count('1'))

    # Statistical features
    total_bits = sum(basic_bits)
    max_run = max(runs) if runs else 0

    # Combine all features
    features = basic_bits + bytes_values + runs + region_counts + parity_bits + transitions + [total_bits, max_run]

    # Ensure fixed length
    return features[:256]  # Limit to 256 features for consistent input size

# Function to train the neural network model with advanced techniques
def train_model(epochs=10, batch_size=128):
    global model_trained

    if len(training_data) < 1000:
        print("Not enough training data yet. Skipping training.")
        return False

    print(f"Training model on {len(training_data)} examples...")

    # Prepare training data
    X = []
    y = []

    # Include all training data
    for features, distance in training_data:
        X.append(features)
        y.append(distance)

    # Add all best examples (rare low distance cases)
    for features, distance in best_examples:
        X.append(features)
        y.append(distance)

    # Convert to numpy for preprocessing
    X = np.array(X)
    y = np.array(y)

    # Normalize features
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    # Convert to PyTorch tensors
    X = torch.tensor(X, dtype=torch.float32).to(device)
    y = torch.tensor(y, dtype=torch.float32).view(-1, 1).to(device)

    # Create dataset and dataloader for batching
    dataset = torch.utils.data.TensorDataset(X, y)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

    # Training loop with validation
    predictor.train()
    best_loss = float('inf')

    for epoch in range(epochs):
        epoch_loss = 0
        num_batches = 0

        for batch_X, batch_y in dataloader:
            optimizer.zero_grad()
            outputs = predictor(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            num_batches += 1

        avg_loss = epoch_loss / num_batches
        print(f'Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}')

        # Update learning rate scheduler
        scheduler.step(avg_loss)

        # Save best model
        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save(predictor.state_dict(), 'best_hamming_predictor.pt')

    # Load best model
    predictor.load_state_dict(torch.load('best_hamming_predictor.pt'))
    model_trained = True

    # Report current learning rate
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Current learning rate: {current_lr}")

    return True

# Function to predict Hamming distance using the neural network
def predict_hamming(key_ints, batch_size=10000):
    """Use the neural network to predict Hamming distances for a batch of keys"""
    predictor.eval()

    all_predictions = []

    # Process in batches to avoid GPU memory issues
    for i in range(0, len(key_ints), batch_size):
        batch = key_ints[i:i+batch_size]
        features = [extract_advanced_features(key) for key in batch]

        # Normalize features
        features_np = np.array(features)
        scaler = StandardScaler()
        features_scaled = scaler.fit_transform(features_np)

        X = torch.tensor(features_scaled, dtype=torch.float32).to(device)

        with torch.no_grad():
            predictions = predictor(X).cpu().numpy().flatten()

        all_predictions.extend(predictions)

    return all_predictions

# Function to compute Bitcoin address from private key
def private_key_to_address(private_key_hex, compressed=False):
    # Convert hex private key to bytes
    private_key_bytes = bytes.fromhex(private_key_hex.zfill(64))

    # Get public key from private key
    signing_key = ecdsa.SigningKey.from_string(private_key_bytes, curve=ecdsa.SECP256k1)
    verifying_key = signing_key.get_verifying_key()

    # Get public key in bytes form
    if compressed:
        if verifying_key.pubkey.point.y() % 2 == 0:
            public_key_bytes = b'\x02' + verifying_key.pubkey.point.x().to_bytes(32, 'big')
        else:
            public_key_bytes = b'\x03' + verifying_key.pubkey.point.x().to_bytes(32, 'big')
    else:
        public_key_bytes = b'\x04' + verifying_key.pubkey.point.x().to_bytes(32, 'big') + verifying_key.pubkey.point.y().to_bytes(32, 'big')

    # Compute SHA256 hash of public key
    sha256_hash = hashlib.sha256(public_key_bytes).digest()

    # Compute RIPEMD160 hash of SHA256 hash
    ripemd160_hash = RIPEMD160.new(sha256_hash).digest()

    # Add version byte (0x00 for mainnet, 0x6f for testnet)
    versioned_hash = b'\x00' + ripemd160_hash

    # Compute checksum (first 4 bytes of double SHA256 of versioned hash)
    checksum = hashlib.sha256(hashlib.sha256(versioned_hash).digest()).digest()[:4]

    # Combine versioned hash and checksum
    binary_address = versioned_hash + checksum

    # Convert to Base58 encoding
    address = base58.b58encode(binary_address).decode('utf-8')

    return address, ripemd160_hash

# GPU-accelerated Hamming distance calculation
def batch_hamming_distance_gpu(keys_batch, hashes_batch, batch_size=1000):
    """
    Calculate Hamming distances for a batch of keys against their hashes using GPU

    Args:
        keys_batch: List of key byte arrays (each 20 bytes)
        hashes_batch: List of hash160 byte arrays (each 20 bytes)
        batch_size: Number of calculations to perform in each GPU batch

    Returns:
        List of Hamming distances
    """
    # Convert byte arrays to bit arrays
    keys_bits = []
    hashes_bits = []

    for i in range(len(keys_batch)):
        # Convert each byte to 8 bits
        key_bits = []
        hash_bits = []

        for j in range(20):  # 20 bytes
            key_byte = keys_batch[i][j] if j < len(keys_batch[i]) else 0
            hash_byte = hashes_batch[i][j] if j < len(hashes_batch[i]) else 0

            for k in range(8):  # 8 bits per byte
                key_bits.append((key_byte >> k) & 1)
                hash_bits.append((hash_byte >> k) & 1)

        keys_bits.append(key_bits)
        hashes_bits.append(hash_bits)

    # Convert to PyTorch tensors
    keys_tensor = torch.tensor(keys_bits, dtype=torch.uint8, device=device)
    hashes_tensor = torch.tensor(hashes_bits, dtype=torch.uint8, device=device)

    # XOR the tensors to get differences
    diff_tensor = torch.bitwise_xor(keys_tensor, hashes_tensor)

    # Sum along bit dimension to get Hamming distances
    distances = torch.sum(diff_tensor, dim=1).cpu().numpy()

    return distances

# Function to calculate Hamming distance between private key and RIPEMD160 hash
def calculate_key_hash_distance(private_key_hex, compressed=False):
    # Get private key in bytes
    if len(private_key_hex) < 64:
        private_key_hex = private_key_hex.zfill(64)
    private_key_bytes = bytes.fromhex(private_key_hex)

    # Ensure private key is 32 bytes (hash160 is 20 bytes)
    private_key_bytes = private_key_bytes[-20:]  # Use last 20 bytes for comparison

    # Get Bitcoin address and RIPEMD160 hash
    address, hash160 = private_key_to_address(private_key_hex, compressed)

    # Calculate Hamming distance on CPU (for individual calculations)
    distance = 0
    for b1, b2 in zip(private_key_bytes, hash160):
        xor = b1 ^ b2
        # Count bits in xor result
        while xor:
            distance += xor & 1
            xor >>= 1

    # Calculate percentage
    percentage = 100 - (distance * 0.625)

    # Add to training data
    key_int = int(private_key_hex, 16)
    features = extract_advanced_features(key_int)

    # Store in training data
    training_data.append((features, distance))

    # If distance is low, store in best examples
    if distance <= 45:  # Keep all examples with distance <= 45
        best_examples.append((features, distance))

    return address, private_key_hex, distance, percentage, hash160

# Improved multi-level screening with curriculum learning
def multi_level_screening(start_int, count, threshold=36, exploration_rate=0.1):
    """
    Use an improved multi-level approach to find promising keys:
    1. Enhanced bit-pattern heuristic screening with exploration
    2. Neural network prediction for pre-screened candidates
    3. Full calculation for most promising candidates
    """
    # Dynamically adjust threshold based on best seen examples
    dynamic_threshold = threshold
    if best_examples:
        # Use a threshold that's slightly higher than the best seen so far
        best_distances = [dist for _, dist in best_examples]
        min_distance = min(best_distances)
        dynamic_threshold = max(threshold, min_distance + 3)

    # Handle very small counts gracefully
    if count <= 0:
        return []

    # 1. Generate candidate keys
    key_ints = list(range(start_int, start_int + count))

    # Optimize for small batches
    if count < 100:
        return key_ints  # Just return all keys for very small batches

    # 2. Initial fast screening (enhanced bit pattern heuristics)
    # Convert integers to tensors for GPU processing
    key_tensors = torch.tensor(key_ints, dtype=torch.int64, device=device)

    # Extract multiple bit patterns
    features = torch.zeros((len(key_ints), 4), device=device)

    # Count bits in different regions of the key
    for i in range(4):
        start_bit = i * 16
        end_bit = (i + 1) * 16
        for j in range(start_bit, end_bit):
            features[:, i] += (key_tensors >> j) & 1

    # Select keys with promising bit patterns
    # Add randomness for exploration
    random_tensor = torch.rand(len(key_ints), device=device)
    exploration_mask = random_tensor < exploration_rate

    # Basic heuristic: keys with moderate number of bits set
    heuristic_mask = (features[:, 0] >= 6) & (features[:, 0] <= 10) & \
                    (features[:, 1] >= 6) & (features[:, 1] <= 10) & \
                    (features[:, 2] >= 6) & (features[:, 2] <= 10) & \
                    (features[:, 3] >= 6) & (features[:, 3] <= 10)

    # Combine masks: either passes heuristic or is randomly selected for exploration
    combined_mask = heuristic_mask | exploration_mask
    first_level_candidates = key_tensors[combined_mask].cpu().numpy()

    # Make sure we have at least some candidates
    if len(first_level_candidates) == 0:
        # Fallback to random selection
        random_indices = torch.randperm(len(key_ints))[:max(100, count // 100)]
        first_level_candidates = key_tensors[random_indices].cpu().numpy()

    print(f"Level 1 screening: {len(first_level_candidates)} candidates out of {count} ({len(first_level_candidates)/count*100:.4f}%)")

    # 3. Neural network prediction (if model has been trained)
    if model_trained and len(first_level_candidates) > 0:
        try:
            # Use neural network for prediction
            predictions = predict_hamming(first_level_candidates)

            # Select keys predicted to have low Hamming distance
            candidates_with_predictions = list(zip(first_level_candidates, predictions))
            candidates_with_predictions.sort(key=lambda x: x[1])  # Sort by predicted distance

            # Adaptive selection rate based on model confidence
            # As training progresses, we become more selective
            if len(best_examples) < 10:
                # Early phase: take more candidates to explore
                top_percentage = 0.2  # 20%
            elif len(best_examples) < 100:
                # Mid phase: moderate selectivity
                top_percentage = 0.1  # 10%
            else:
                # Late phase: high selectivity
                top_percentage = 0.05  # 5%

            # Always take at least some candidates
            top_count = max(10, int(len(first_level_candidates) * top_percentage))
            second_level_candidates = [x[0] for x in candidates_with_predictions[:top_count]]

            print(f"Level 2 screening: {len(second_level_candidates)} candidates out of {len(first_level_candidates)} ({len(second_level_candidates)/len(first_level_candidates)*100:.4f}%)")
        except Exception as e:
            # If prediction fails, just use first level candidates
            print(f"Level 2 screening failed with error: {e}")
            second_level_candidates = first_level_candidates
            print("Using Level 1 candidates instead")
    else:
        # If model isn't trained yet, skip this level
        second_level_candidates = first_level_candidates
        print("Skipping Level 2 screening (model not trained yet)")

    return second_level_candidates

# GPU-accelerated batch processing with improved screening
def process_keys_gpu(target_addresses, start_int, batch_size, threshold=36):
    """Process a batch of keys using GPU acceleration and improved screening"""
    results = []

    # Use multi-level screening to find promising candidates
    promising_keys = multi_level_screening(start_int, batch_size, threshold)

    # Process promising keys in smaller GPU batches
    sub_batch_size = 1000  # Process this many at a time on GPU
    for i in range(0, len(promising_keys), sub_batch_size):
        sub_batch = promising_keys[i:i+sub_batch_size]

        # Generate key bytes and calculate addresses, hashes
        key_bytes_list = []
        address_list = []
        hash160_list = []

        for key_int in sub_batch:
            key_hex = format(key_int, 'x').zfill(64)
            key_bytes = bytes.fromhex(key_hex)[-20:]  # Last 20 bytes

            # Calculate address and hash160
            address, hash160 = private_key_to_address(key_hex, False)  # Uncompressed

            key_bytes_list.append(key_bytes)
            address_list.append(address)
            hash160_list.append(hash160)

        # Calculate Hamming distances using GPU
        distances = batch_hamming_distance_gpu(key_bytes_list, hash160_list)

        # Process results
        for j in range(len(sub_batch)):
            key_int = sub_batch[j]
            key_hex = format(key_int, 'x').zfill(64)
            address = address_list[j]
            distance = distances[j]
            percentage = 100 - (distance * 0.625)

            # Add to training data
            features = extract_advanced_features(key_int)
            training_data.append((features, distance))

            # Add to best examples if distance is low
            if distance <= 45:
                best_examples.append((features, distance))

                # Report this good example
                with print_lock:
                    print(f"Found good example: Address={address}, Distance={distance}, Percentage={percentage:.2f}%")

            # Check if Hamming distance meets threshold
            if distance <= threshold:
                result = (address, key_hex, int(distance), percentage, 0.0, False, key_int)

                # Check if in target list
                if address in target_addresses:
                    results.append(result)

                # Update global best
                with print_lock:
                    if address not in global_bests or distance < global_bests[address][2]:
                        global_bests[address] = result
                        print(f"NEW BEST for {address}: Key={key_hex}, Distance={distance}, Percentage={percentage:.2f}%, Integer={key_int}")

    return results

# Function to check a batch of keys using multi-threading and GPU acceleration
def check_addresses_batch(target_addresses, private_key_int, start_idx, end_idx, threshold=36, chunk_id=0):
    # Initialize stats
    keys_screened = end_idx - start_idx

    # Calculate batch size for GPU processing
    gpu_batch_size = 100000
    results = []

    # Process in GPU batches
    for batch_start in range(start_idx, end_idx, gpu_batch_size):
        batch_end = min(batch_start + gpu_batch_size, end_idx)
        batch_size = batch_end - batch_start

        # Process this batch
        batch_results = process_keys_gpu(
            target_addresses,
            private_key_int + batch_start,
            batch_size,
            threshold
        )

        results.extend(batch_results)

        # Train model periodically
        global_training_size = len(training_data) + len(best_examples)
        if global_training_size >= 10000 and (batch_start // gpu_batch_size) % 5 == 0:
            train_model(epochs=10, batch_size=256)

    # Log progress
    with print_lock:
        print(f"Chunk {chunk_id}: Screened {keys_screened} keys, found {len(results)} results")

    return results, keys_screened, len(results)

# Function to load target addresses from previous data
def load_target_addresses_from_data(data_string):
    addresses = []
    lines = data_string.strip().split('\n')
    for line in lines:
        parts = line.split('\t')
        if len(parts) >= 1:
            addresses.append(parts[0])
    return addresses

# Function to load previous results to set initial global bests
def load_previous_results(data_string):
    lines = data_string.strip().split('\n')
    for line in lines:
        parts = line.split('\t')
        if len(parts) >= 6:
            address = parts[0]
            key_hex = parts[1]
            try:
                distance = int(parts[2])
                percentage = float(parts[3])
                entropy = float(parts[4])
                compressed = (parts[5] == 'True')
                # Convert hex key to integer
                current_key = int(key_hex, 16)

                # Store in global bests
                global_bests[address] = (address, key_hex, distance, percentage, entropy, compressed, current_key)
                print(f"Loaded previous best for {address}: Distance={distance}, Percentage={percentage:.2f}%, Integer={current_key}")

                # Add to training data for neural network
                features = extract_advanced_features(current_key)
                training_data.append((features, distance))

                # Add to best examples if distance is low
                if distance <= 45:
                    best_examples.append((features, distance))

            except (ValueError, IndexError) as e:
                print(f"Warning: Couldn't parse line: {line}")

    # Train initial model if enough data is loaded
    if len(training_data) >= 1000:
        train_model(epochs=20, batch_size=256)

    return global_bests

# Main function to run the search with improved GPU acceleration and neural network
def run_search(target_addresses, start_from_int, num_keys_to_check, threshold=36, num_threads=8, batch_size=1000000):
    print(f"Starting GPU-accelerated search with enhanced neural network assistance")
    print(f"Using {num_threads} threads from integer {start_from_int}, checking {num_keys_to_check} keys")
    print(f"Using threshold < {threshold} for Hamming distance")
    print(f"Keeping up to {training_data.maxlen} training examples and unlimited best examples")

    all_results = []
    start_time = time.time()
    batch_counter = 0

    # Stats tracking
    total_keys_screened = 0
    total_keys_processed = 0

    with concurrent.futures.ThreadPoolExecutor(max_workers=num_threads) as executor:
        for batch_start in range(0, num_keys_to_check, batch_size):
            batch_counter += 1
            batch_end = min(batch_start + batch_size, num_keys_to_check)

            # Split batch into chunks for each thread
            chunk_size = (batch_end - batch_start) // num_threads
            futures = []

            for i in range(num_threads):
                chunk_start = batch_start + i * chunk_size
                chunk_end = batch_start + (i + 1) * chunk_size if i < num_threads - 1 else batch_end

                # Submit chunk to thread pool
                future = executor.submit(
                    check_addresses_batch,
                    target_addresses,
                    start_from_int,
                    chunk_start,
                    chunk_end,
                    threshold,
                    f"B{batch_counter}-T{i}"
                )
                futures.append(future)

            # Collect results from threads
            batch_results = []
            batch_screened = 0
            batch_processed = 0

            for future in concurrent.futures.as_completed(futures):
                results, keys_screened, keys_processed = future.result()
                batch_results.extend(results)
                batch_screened += keys_screened
                batch_processed += keys_processed

            all_results.extend(batch_results)

            # Update stats
            total_keys_screened += batch_screened
            total_keys_processed += batch_processed

            # Report progress
            elapsed = time.time() - start_time
            keys_checked = batch_end
            keys_per_second = keys_checked / elapsed if elapsed > 0 else 0

            print(f"\n--- PROGRESS REPORT ---")
            print(f"Keys checked: {keys_checked:,}/{num_keys_to_check:,} ({keys_checked/num_keys_to_check*100:.5f}%)")
            print(f"Speed: {keys_per_second:.2f} keys/second, Elapsed: {elapsed:.2f} seconds")
            print(f"Found results: {total_keys_processed:,}")
            print(f"Neural network training examples: {len(training_data)}")
            print(f"Best examples (distance ≤ 45): {len(best_examples)}")
            print(f"Current global bests: {len(global_bests)} addresses with improved Hamming distances")

            # Show statistics about training data
            if training_data:
                distances = [dist for _, dist in training_data]
                min_dist = min(distances) if distances else 0
                max_dist = max(distances) if distances else 0
                avg_dist = sum(distances) / len(distances) if distances else 0
                print(f"Training data stats: Min={min_dist}, Max={max_dist}, Avg={avg_dist:.2f}")

            # Show top 5 best results so far
            print("\n--- TOP 5 BEST RESULTS ---")
            if global_bests:
                sorted_bests = sorted(global_bests.items(), key=lambda x: x[1][2])
                for i, (address, (addr, key_hex, distance, percentage, entropy, compressed, integer)) in enumerate(sorted_bests[:5]):
                    print(f"{i+1}. {addr}: Distance={distance}, Percentage={percentage:.2f}%, Integer={integer}")

            # Save intermediate results
            if batch_counter % 5 == 0:
                save_results_to_csv(all_results, 'intermediate_results.csv')
                save_global_bests_to_csv(global_bests, 'global_bests.csv')

                # Save model
                torch.save({
                    'model_state_dict': predictor.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'scheduler_state_dict': scheduler.state_dict(),
                    'best_examples': best_examples[:1000]  # Save top 1000 best examples
                }, 'hamming_predictor.pt')

                # Clear accumulated results to save memory
                all_results = []

    # Final save
    save_results_to_csv(all_results, 'final_results.csv')
    save_global_bests_to_csv(global_bests, 'global_bests.csv')

    # Save final model with data
    torch.save({
        'model_state_dict': predictor.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_examples': best_examples
    }, 'hamming_predictor_final.pt')

    return all_results

# Function to save results to CSV
def save_results_to_csv(results, filename):
    if not results:
        print(f"No results to save to {filename}")
        return

    df = pd.DataFrame(results, columns=['Address', 'PrivateKeyHex', 'HammingDistance', 'Percentage', 'Entropy', 'Compressed', 'Integer'])
    df.to_csv(filename, index=False)
    print(f"Saved {len(results)} results to {filename}")

# Function to save global bests to CSV
def save_global_bests_to_csv(bests, filename):
    if not bests:
        print(f"No global bests to save to {filename}")
        return

    data = []
    for address, (addr, key_hex, distance, percentage, entropy, compressed, integer) in bests.items():
        data.append([addr, key_hex, distance, percentage, entropy, compressed, integer])

    df = pd.DataFrame(data, columns=['Address', 'PrivateKeyHex', 'HammingDistance', 'Percentage', 'Entropy', 'Compressed', 'Integer'])
    df.to_csv(filename, index=False)
    print(f"Saved {len(bests)} global bests to {filename}")

# Function to save training data and best examples
def save_training_data(filename='training_data.npz'):
    # Convert training data to numpy arrays
    features = []
    distances = []

    for feature, distance in training_data:
        features.append(feature)
        distances.append(distance)

    # Convert best examples too
    best_features = []
    best_distances = []

    for feature, distance in best_examples:
        best_features.append(feature)
        best_distances.append(distance)

    # Save to compressed numpy file
    np.savez_compressed(
        filename,
        features=np.array(features),
        distances=np.array(distances),
        best_features=np.array(best_features),
        best_distances=np.array(best_distances)
    )

    print(f"Saved {len(features)} training examples and {len(best_features)} best examples to {filename}")

    # Make available for download
    files.download(filename)

# Function to load saved training data
def load_training_data(filename):
    data = np.load(filename)

    # Load regular training data
    features = data['features']
    distances = data['distances']

    for i in range(len(features)):
        training_data.append((features[i], distances[i]))

    # Load best examples
    best_features = data['best_features']
    best_distances = data['best_distances']

    for i in range(len(best_features)):
        best_examples.append((best_features[i], best_distances[i]))

    print(f"Loaded {len(features)} training examples and {len(best_features)} best examples")

# Upload function for user to provide the data
def upload_and_start():
    print("Please upload your existing data file (mthfr.txt or similar):")
    uploaded = files.upload()

    if not uploaded:
        print("No file uploaded. Using empty target list.")
        target_addresses = []
    else:
        file_name = list(uploaded.keys())[0]
        data_string = uploaded[file_name].decode('utf-8')
        target_addresses = load_target_addresses_from_data(data_string)
        load_previous_results(data_string)

        print(f"Loaded {len(target_addresses)} target addresses.")

    # Ask if there's a saved training data file to load
    load_saved_data = input("Do you have a saved training data file to load? (y/n): ")
    if load_saved_data.lower() == 'y':
        print("Please upload your training data file:")
        training_data_uploaded = files.upload()
        if training_data_uploaded:
            train_file_name = list(training_data_uploaded.keys())[0]
            load_training_data(train_file_name)

    # Ask if there's a saved model to load
    load_saved_model = input("Do you have a saved model to load? (y/n): ")
    if load_saved_model.lower() == 'y':
        print("Please upload your model file:")
        model_uploaded = files.upload()
        if model_uploaded:
            model_file_name = list(model_uploaded.keys())[0]
            checkpoint = torch.load(model_file_name)
            predictor.load_state_dict(checkpoint['model_state_dict'])
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

            # Optionally load best examples from the model file
            if 'best_examples' in checkpoint:
                global best_examples
                best_examples = checkpoint['best_examples']
                print(f"Loaded {len(best_examples)} best examples from model file")

            global model_trained
            model_trained = True
            print("Model loaded successfully!")

    # Get user input for search parameters
    start_from_str = input("Enter starting integer (e.g., 0, 1000000): ")
    start_from_int = int(start_from_str)

    num_keys_str = input("Enter number of keys to check (e.g., 1000000000): ")
    num_keys_to_check = int(num_keys_str)

    threshold_str = input("Enter Hamming distance threshold (default 36): ")
    threshold = int(threshold_str) if threshold_str else 36

    num_threads_str = input("Enter number of threads to use (default 8 for GPU coordination): ")
    num_threads = int(num_threads_str) if num_threads_str else 8

    batch_size_str = input("Enter batch size (default 1,000,000): ")
    batch_size = int(batch_size_str) if batch_size_str else 1000000

    # Run the search
    results = run_search(target_addresses, start_from_int, num_keys_to_check, threshold, num_threads, batch_size)

    # Print final summary
    print("\nSearch completed!")
    print(f"Total matches found: {len(results)}")
    print("Global bests:")
    for address, (addr, key_hex, distance, percentage, entropy, compressed, integer) in sorted(global_bests.items(), key=lambda x: x[1][2]):
        print(f"{addr}: Distance={distance}, Percentage={percentage:.2f}%, Key={key_hex}, Integer={integer}")

    # Save and download results
    save_results_to_csv(results, 'final_results.csv')
    save_global_bests_to_csv(global_bests, 'global_bests.csv')
    save_training_data('training_data.npz')

    # Download final files
    files.download('global_bests.csv')
    files.download('final_results.csv')
    files.download('hamming_predictor_final.pt')
    files.download('training_data.npz')

# GPU warming up - small initial calculation to initialize CUDA
def warm_up_gpu():
    if torch.cuda.is_available():
        print("Warming up GPU...")
        x = torch.zeros(1000, 1000, device=device)
        y = torch.matmul(x, x)
        torch.cuda.synchronize()
        print("GPU ready!")

# Start with GPU warmup
warm_up_gpu()

# Start the program
upload_and_start()

Using device: cuda
GPU: NVIDIA L4
GPU Memory: 23.80 GB
Warming up GPU...
GPU ready!
Please upload your existing data file (mthfr.txt or similar):


Saving mthfr.txt to mthfr (1).txt
Loaded previous best for 1CVyPG2eMjQhFiS84r1eddFjhCfu4RfmDS: Distance=44, Percentage=72.50%, Integer=94298207
Loaded previous best for 1PynTQMsFzAQH411CYctGmwuiCgBgaaFbi: Distance=41, Percentage=74.38%, Integer=17841
Loaded previous best for 1JSxDnLYD4XKTQ73N7in2M9XRovw5LANiu: Distance=46, Percentage=71.25%, Integer=107157515
Loaded previous best for 1273UMTmzwHBruimCnepy7d9oP1rhRx7YK: Distance=47, Percentage=70.62%, Integer=53342
Loaded previous best for 1FgiDfz7jmdjTUaJ9GsKbNbgsmu7T31Dvx: Distance=47, Percentage=70.62%, Integer=117183000
Loaded previous best for 17pPrzDUtwebfhB15UQnHmJnMAj58cLEBu: Distance=45, Percentage=71.88%, Integer=189175421
Loaded previous best for 1DiZTFtVLEXFBHXify2qThAPkE6dMKViXH: Distance=48, Percentage=70.00%, Integer=7106630
Loaded previous best for 1Mnc9YWgfSyyut8wdKDNznYhi2NvYnsqCw: Distance=47, Percentage=70.62%, Integer=9273048
Loaded previous best for 1HYDYuWDptZp3sDLkqek3gZYGnqjRKZumu: Distance=45, Percentage=71.88%

RuntimeError: mat1 and mat2 shapes cannot be multiplied (256x129 and 256x512)